# Netflix Content Recommendation System
**Internship Task 1:** Content-based recommendations with TF-IDF and cosine similarity.

The CSV contains content metadata, not user watch history. Genre overlap is a proxy evaluation metric.

## 1. Import libraries and load CSV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

path = Path("../data/Dataset.csv")
if not path.exists():
    path = Path("data/Dataset.csv")
df = pd.read_csv(path)
print("Shape:", df.shape)
display(df.head())

## 2. Data exploration

In [ ]:
display(df.info())
display(df.isna().sum().to_frame("missing"))
print("Duplicate title/type pairs:", df.duplicated(["title", "type"]).sum())
display(df["type"].value_counts())

## 3. Basic visualization

In [ ]:
df["type"].value_counts().plot(kind="bar", title="Netflix content types")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

## 4. Feature engineering
We double-weight genre text, then add director, country, type, and rating.

In [ ]:
import re
def clean_tokens(text):
    return re.sub(r"[^a-z0-9 ]+", " ", str(text).lower()).strip()

data = df.drop_duplicates(["title", "type"]).reset_index(drop=True).copy()
cols = ["title", "type", "director", "country", "rating", "listed_in"]
data[cols] = data[cols].fillna("").astype(str)
data["combined_features"] = (
    data["listed_in"].map(clean_tokens) + " " + data["listed_in"].map(clean_tokens) + " " +
    data["director"].map(clean_tokens) + " " + data["country"].map(clean_tokens) + " " +
    data["type"].map(clean_tokens) + " " + data["rating"].map(clean_tokens)
)
display(data[["title", "combined_features"]].head())

## 5. TF-IDF text vectorization

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(data["combined_features"])
print("Sparse TF-IDF matrix:", X.shape)

## 6. Cosine similarity and recommendations
Compute similarity on demand instead of allocating a large all-pairs matrix.

In [ ]:
def recommend(title, top_n=5):
    matches = data.index[data["title"].str.casefold() == title.strip().casefold()].tolist()
    if not matches:
        raise ValueError(f"Title not found: {title}")
    idx = matches[0]
    scores = cosine_similarity(X[idx], X).ravel()
    scores[idx] = -1
    best = np.argsort(-scores, kind="stable")[:top_n]
    output = data.iloc[best][["title", "type", "listed_in"]].copy()
    output["similarity_score"] = np.round(scores[best], 3)
    return output.reset_index(drop=True)

display(recommend("Ganglands", 5))

## 7. Evaluate genre relevance (Jaccard overlap)
This is a proxy metric, not a ground-truth accuracy score.

In [ ]:
def genre_set(s):
    return {g.strip().casefold() for g in s.split(",") if g.strip()}

def evaluate(title, n=5):
    original = data.loc[data["title"].str.casefold() == title.casefold(), "listed_in"].iloc[0]
    a = genre_set(original)
    results = recommend(title, n)
    results["genre_overlap"] = [len(a & genre_set(s)) / len(a | genre_set(s)) if a | genre_set(s) else 0 for s in results["listed_in"]]
    print("Average genre overlap:", round(results["genre_overlap"].mean(), 3))
    return results

results = evaluate("Ganglands")
display(results)

## 8. Visualize similarity scores

In [ ]:
results.sort_values("similarity_score").plot.barh(x="title", y="similarity_score", legend=False, figsize=(9, 5))
plt.title("Top 5 content-based recommendations for Ganglands")
plt.xlabel("Cosine similarity")
plt.tight_layout()
plt.show()

## 9. Conclusion
The system provides similar titles based on content attributes. Limitations: no plot descriptions, posters, or user feedback, so recommendations cannot be evaluated with real user satisfaction or accuracy without additional data.